In [1]:
import pandas as pd
import numpy as np

try:
    feature_df = eda_df.copy()
    print("Using EDA dataset")
except NameError:
    feature_df = pd.read_excel(
        "cleaned_personalized_healthcare_data.xlsx",
        sheet_name="Cleaned_Data"
    )
    print("Using saved cleaned dataset")

print("Dataset shape:", feature_df.shape)
print(feature_df.columns.tolist())

feature_df.head()

Using saved cleaned dataset
Dataset shape: (500, 15)
['patient_id', 'age_years', 'sex', 'condition', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'historical_medication_class', 'recorded_outcome', 'data_quality_status']


,patient_id,age_years,sex,condition,glucose_mg_dl,systolic_bp_mmhg,diastolic_bp_mmhg,cholesterol_mg_dl,heart_rate_bpm,recorded_allergy,family_history,adherence_level,historical_medication_class,recorded_outcome,data_quality_status
0,SYN-0001,28,Female,Seasonal Allergy,97,132,85,214,73,None recorded,No,High,Antihistamine class A,Improved,Valid
1,SYN-0002,80,Female,Acid Reflux,80,123,79,189,59,None recorded,Yes,Medium,Acid-suppression class A,Improved,Valid
2,SYN-0003,36,Female,Asthma,86,121,66,192,82,None recorded,No,Medium,Controller inhaler class,Follow-up required,Valid
3,SYN-0004,21,Male,High Cholesterol,103,120,69,278,85,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Valid
4,SYN-0005,58,Male,High Cholesterol,79,111,76,244,64,None recorded,Yes,High,Lipid-lowering class B,Follow-up required,Valid


In [2]:
feature_df.columns = (
    feature_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

feature_df.columns.tolist()

['patient_id',
 'age_years',
 'sex',
 'condition',
 'glucose_mg_dl',
 'systolic_bp_mmhg',
 'diastolic_bp_mmhg',
 'cholesterol_mg_dl',
 'heart_rate_bpm',
 'recorded_allergy',
 'family_history',
 'adherence_level',
 'historical_medication_class',
 'recorded_outcome',
 'data_quality_status']

In [3]:
feature_df["age_group"] = pd.cut(
    feature_df["age_years"],
    bins=[17, 29, 44, 59, 74, 120],
    labels=[
        "18-29",
        "30-44",
        "45-59",
        "60-74",
        "75+"
    ],
    include_lowest=True
)

feature_df[
    ["age_years", "age_group"]
].head()

,age_years,age_group
0,28,18-29
1,80,75+
2,36,30-44
3,21,18-29
4,58,45-59


In [4]:
feature_df["pulse_pressure_mmhg"] = (
    feature_df["systolic_bp_mmhg"] -
    feature_df["diastolic_bp_mmhg"]
)

feature_df["systolic_diastolic_ratio"] = (
    feature_df["systolic_bp_mmhg"] /
    feature_df["diastolic_bp_mmhg"].replace(0, np.nan)
)

feature_df[
    [
        "systolic_bp_mmhg",
        "diastolic_bp_mmhg",
        "pulse_pressure_mmhg",
        "systolic_diastolic_ratio"
    ]
].head()

,systolic_bp_mmhg,diastolic_bp_mmhg,pulse_pressure_mmhg,systolic_diastolic_ratio
0,132,85,47,1.552941
1,123,79,44,1.556962
2,121,66,55,1.833333
3,120,69,51,1.739130
4,111,76,35,1.460526


In [5]:
feature_df["glucose_cholesterol_interaction"] = (
    feature_df["glucose_mg_dl"] *
    feature_df["cholesterol_mg_dl"]
)

feature_df["glucose_age_interaction"] = (
    feature_df["glucose_mg_dl"] *
    feature_df["age_years"]
)

feature_df["cholesterol_age_interaction"] = (
    feature_df["cholesterol_mg_dl"] *
    feature_df["age_years"]
)

feature_df["bp_age_interaction"] = (
    feature_df["systolic_bp_mmhg"] *
    feature_df["age_years"]
)

feature_df[
    [
        "glucose_cholesterol_interaction",
        "glucose_age_interaction",
        "cholesterol_age_interaction",
        "bp_age_interaction"
    ]
].head()

,glucose_cholesterol_interaction,glucose_age_interaction,cholesterol_age_interaction,bp_age_interaction
0,20758,2716,5992,3696
1,15120,6400,15120,9840
2,16512,3096,6912,4356
3,28634,2163,5838,2520
4,19276,4582,14152,6438


In [6]:
feature_df["allergy_recorded_flag"] = np.where(
    feature_df["recorded_allergy"]
    .str.strip()
    .str.lower()
    .eq("none recorded"),
    0,
    1
)

feature_df[
    ["recorded_allergy", "allergy_recorded_flag"]
].head(10)

,recorded_allergy,allergy_recorded_flag
0,None recorded,0
1,None recorded,0
2,None recorded,0
3,None recorded,0
4,None recorded,0
5,None recorded,0
6,None recorded,0
7,Sulfonamide,1
8,None recorded,0
9,None recorded,0


In [7]:
family_history_mapping = {
    "No": 0,
    "Yes": 1
}

feature_df["family_history_flag"] = (
    feature_df["family_history"]
    .map(family_history_mapping)
)

feature_df[
    ["family_history", "family_history_flag"]
].head()

,family_history,family_history_flag
0,No,0
1,Yes,1
2,No,0
3,Yes,1
4,Yes,1


In [8]:
adherence_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

feature_df["adherence_score"] = (
    feature_df["adherence_level"]
    .map(adherence_mapping)
)

feature_df[
    ["adherence_level", "adherence_score"]
].head()

,adherence_level,adherence_score
0,High,2
1,Medium,1
2,Medium,1
3,High,2
4,High,2


In [9]:
measurement_columns = [
    "glucose_mg_dl",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "cholesterol_mg_dl",
    "heart_rate_bpm"
]

for column in measurement_columns:
    mean_value = feature_df[column].mean()
    standard_deviation = feature_df[column].std()

    feature_df[f"{column}_dataset_zscore"] = (
        (feature_df[column] - mean_value) /
        standard_deviation
    )

zscore_columns = [
    column for column in feature_df.columns
    if column.endswith("_dataset_zscore")
]

feature_df[zscore_columns].head()

,glucose_mg_dl_dataset_zscore,systolic_bp_mmhg_dataset_zscore,diastolic_bp_mmhg_dataset_zscore,cholesterol_mg_dl_dataset_zscore,heart_rate_bpm_dataset_zscore
0,-0.324226,0.254356,0.339338,0.424061,-0.315852
1,-0.809337,-0.237734,-0.210049,-0.216205,-1.546215
2,-0.638121,-0.347087,-1.400387,-0.139373,0.475096
3,-0.153010,-0.401764,-1.125693,2.063142,0.738745
4,-0.837873,-0.893854,-0.484742,1.192380,-1.106799


In [10]:
feature_df["measurement_deviation_score"] = (
    feature_df[zscore_columns]
    .abs()
    .mean(axis=1)
)

feature_df[
    [
        "patient_id",
        "measurement_deviation_score"
    ]
].head()

,patient_id,measurement_deviation_score
0,SYN-0001,0.331566
1,SYN-0002,0.603908
2,SYN-0003,0.600013
3,SYN-0004,0.896471
4,SYN-0005,0.903130


In [11]:
engineered_columns = [
    "age_group",
    "pulse_pressure_mmhg",
    "systolic_diastolic_ratio",
    "glucose_cholesterol_interaction",
    "glucose_age_interaction",
    "cholesterol_age_interaction",
    "bp_age_interaction",
    "allergy_recorded_flag",
    "family_history_flag",
    "adherence_score",
    "measurement_deviation_score"
]

feature_df[engineered_columns].head()

print(
    "Missing values:",
    feature_df[engineered_columns].isnull().sum().sum()
)

numeric_engineered = feature_df[
    engineered_columns
].select_dtypes(include=np.number)

print(
    "Infinite values:",
    np.isinf(numeric_engineered).sum().sum()
)

Missing values: 0
Infinite values: 0


In [12]:
feature_df = feature_df.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite values replaced with NaN")

feature_df[engineered_columns].isnull().sum()

Infinite values replaced with NaN


age_group                          0
pulse_pressure_mmhg                0
systolic_diastolic_ratio           0
glucose_cholesterol_interaction    0
glucose_age_interaction            0
cholesterol_age_interaction        0
bp_age_interaction                 0
allergy_recorded_flag              0
family_history_flag                0
adherence_score                    0
measurement_deviation_score        0
dtype: int64

In [13]:
target_column = "historical_medication_class"

y = feature_df[target_column].copy()

print("Target:", target_column)
print("Number of target classes:", y.nunique())

print("\nTarget distribution:")
print(y.value_counts())

Target: historical_medication_class
Number of target classes: 12

Target distribution:
historical_medication_class
Antihistamine class A       52
Controller inhaler class    47
Antidiabetic class A        45
Antihypertensive class A    43
Acid-suppression class A    42
Antihypertensive class B    42
Antidiabetic class B        42
Lipid-lowering class A      42
Antihistamine class B       40
Bronchodilator class        36
Acid-suppression class B    35
Lipid-lowering class B      34
Name: count, dtype: int64


In [14]:
dataset_statistical_features = [
    column for column in feature_df.columns
    if column.endswith("_dataset_zscore")
]

dataset_statistical_features.append(
    "measurement_deviation_score"
)

columns_to_exclude = [
    "patient_id",
    "historical_medication_class",
    "recorded_outcome",
    "dataset_notice",
    "data_quality_status"
] + dataset_statistical_features

X = feature_df.drop(
    columns=columns_to_exclude,
    errors="ignore"
).copy()

y = feature_df[
    "historical_medication_class"
].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of target classes:", y.nunique())

print("\nExcluded statistical features:")
print(dataset_statistical_features)

print("\nFinal model features:")
print(X.columns.tolist())

# Final verification
assert X.shape == (500, 21)
assert "measurement_deviation_score" not in X.columns

assert not any(
    column.endswith("_dataset_zscore")
    for column in X.columns
)

print("\nFeature matrix verified successfully.")

Feature matrix shape: (500, 21)
Target shape: (500,)
Number of target classes: 12

Excluded statistical features:
['glucose_mg_dl_dataset_zscore', 'systolic_bp_mmhg_dataset_zscore', 'diastolic_bp_mmhg_dataset_zscore', 'cholesterol_mg_dl_dataset_zscore', 'heart_rate_bpm_dataset_zscore', 'measurement_deviation_score']

Final model features:
['age_years', 'sex', 'condition', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'age_group', 'pulse_pressure_mmhg', 'systolic_diastolic_ratio', 'glucose_cholesterol_interaction', 'glucose_age_interaction', 'cholesterol_age_interaction', 'bp_age_interaction', 'allergy_recorded_flag', 'family_history_flag', 'adherence_score']

Feature matrix verified successfully.


In [15]:
final_numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

final_categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:")
print(final_numeric_features)

print("\nCategorical features:")
print(final_categorical_features)

Numeric features:
['age_years', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'pulse_pressure_mmhg', 'systolic_diastolic_ratio', 'glucose_cholesterol_interaction', 'glucose_age_interaction', 'cholesterol_age_interaction', 'bp_age_interaction', 'allergy_recorded_flag', 'family_history_flag', 'adherence_score']

Categorical features:
['sex', 'condition', 'recorded_allergy', 'family_history', 'adherence_level', 'age_group']


In [16]:
feature_output_file = "clinical_feature_engineered_data.xlsx"

with pd.ExcelWriter(
    feature_output_file,
    engine="openpyxl"
) as writer:

    feature_df.to_excel(
        writer,
        sheet_name="Feature_Engineered_Data",
        index=False
    )

    X.to_excel(
        writer,
        sheet_name="Model_Features",
        index=False
    )

    pd.DataFrame({
        "historical_medication_class": y
    }).to_excel(
        writer,
        sheet_name="Target",
        index=False
    )

print("Feature-engineered dataset saved as:")
print(feature_output_file)

Feature-engineered dataset saved as:
clinical_feature_engineered_data.xlsx
